<a href="https://colab.research.google.com/github/rajilsaj/nasa-mosaics-project/blob/xgboost/notebooks/01_temporal_split.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import pandas as pd
import numpy as np
import os
from google.colab import drive

drive.mount('/content/drive')

# Paths
BASE_PATH = "/content/drive/MyDrive/2026/www/nasa-mosaics-project"

# Check if BASE_PATH exists
if not os.path.exists(BASE_PATH):
    print(f"Warning: BASE_PATH does not exist: {BASE_PATH}")
    print("Please ensure the path is correct in your Google Drive.")
else:
    print(f"BASE_PATH found: {BASE_PATH}")

ML_FILE = f"{BASE_PATH}/data/raw/ml_ready_vortex_data.csv"
JACKSON_FILE = f"{BASE_PATH}/data/raw/Jackson_vortex_detections_reformatted_augmented.csv"
OUTPUT_DIR = f"{BASE_PATH}/data/splits"

# Split configuration
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
GAP_RATIO = 0.03  # 3% temporal gap


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BASE_PATH found: /content/drive/MyDrive/2026/www/nasa-mosaics-project


In [9]:
ml_df = pd.read_csv(ML_FILE)
jackson_df = pd.read_csv(JACKSON_FILE)

print("ML samples:", len(ml_df))
print("Jackson events:", len(jackson_df))


ML samples: 3590168
Jackson events: 309


In [10]:
ml_df = pd.read_csv(ML_FILE)
jackson_df = pd.read_csv(JACKSON_FILE)

print("ML samples:", len(ml_df))
print("Jackson events:", len(jackson_df))

ML samples: 3590168
Jackson events: 309


In [11]:
# Ensure SCLK is numeric
ml_df["SCLK"] = pd.to_numeric(ml_df["SCLK"], errors="coerce")
jackson_df["SCLK"] = pd.to_numeric(jackson_df["SCLK"], errors="coerce")

# Drop invalid rows
ml_df = ml_df.dropna(subset=["SCLK"])
jackson_df = jackson_df.dropna(subset=["SCLK"])

# Sort chronologically
ml_df = ml_df.sort_values("SCLK").reset_index(drop=True)
jackson_df = jackson_df.sort_values("SCLK").reset_index(drop=True)

print("SCLK range:")
print(ml_df["SCLK"].min(), "→", ml_df["SCLK"].max())


SCLK range:
667042464 → 674883874


In [12]:
n = len(ml_df)

train_end = int(n * TRAIN_RATIO)
gap1_end = int(n * (TRAIN_RATIO + GAP_RATIO))
val_end = int(n * (TRAIN_RATIO + GAP_RATIO + VAL_RATIO))
gap2_end = int(n * (TRAIN_RATIO + 2 * GAP_RATIO + VAL_RATIO))

ml_train = ml_df.iloc[:train_end].reset_index(drop=True)
ml_val = ml_df.iloc[gap1_end:val_end].reset_index(drop=True)
ml_test = ml_df.iloc[gap2_end:].reset_index(drop=True)

print("Train:", len(ml_train))
print("Val:", len(ml_val))
print("Test:", len(ml_test))


Train: 2513117
Val: 538525
Test: 323116


In [13]:
def split_jackson(split_df):
    min_sclk = split_df["SCLK"].min()
    max_sclk = split_df["SCLK"].max()
    return jackson_df[
        (jackson_df["SCLK"] >= min_sclk) &
        (jackson_df["SCLK"] <= max_sclk)
    ].reset_index(drop=True)

jackson_train = split_jackson(ml_train)
jackson_val = split_jackson(ml_val)
jackson_test = split_jackson(ml_test)

print("Jackson Train:", len(jackson_train))
print("Jackson Val:", len(jackson_val))
print("Jackson Test:", len(jackson_test))


Jackson Train: 225
Jackson Val: 46
Jackson Test: 23


In [14]:
import shutil

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Delete existing files in OUTPUT_DIR
for filename in os.listdir(OUTPUT_DIR):
    file_path = os.path.join(OUTPUT_DIR, filename)
    try:
        if os.path.isfile(file_path) or os.path.islink(file_path):
            os.unlink(file_path)
        elif os.path.isdir(file_path):
            shutil.rmtree(file_path)
    except Exception as e:
        print(f'Failed to delete {file_path}. Reason: {e}')

ml_train.to_csv(f"{OUTPUT_DIR}/ml_train.csv", index=False)
ml_val.to_csv(f"{OUTPUT_DIR}/ml_val.csv", index=False)
ml_test.to_csv(f"{OUTPUT_DIR}/ml_test.csv", index=False)

jackson_train.to_csv(f"{OUTPUT_DIR}/jackson_train.csv", index=False)
jackson_val.to_csv(f"{OUTPUT_DIR}/jackson_val.csv", index=False)
jackson_test.to_csv(f"{OUTPUT_DIR}/jackson_test.csv", index=False)

print("✅ Splits saved to data/splits/")

✅ Splits saved to data/splits/
